# Análise — Previsor de Incêndios no RN

Notebook complementar ao projeto. Examina o dataset construído pelo ETL, a distribuição de focos, correlações com meteorologia e o desempenho dos modelos.

Pré-requisitos: a pipeline `make tudo` já rodou (`dados/processados/fato_municipio_dia.parquet` e `backend/modelo/avaliacao/metricas.json` existem).

In [ ]:
import os, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# garante que o backend seja importavel
RAIZ = os.path.abspath('..')
sys.path.insert(0, RAIZ)

df = pd.read_parquet(os.path.join(RAIZ, 'dados/processados/fato_municipio_dia.parquet'))
df['data'] = pd.to_datetime(df['data'])
print('shape:', df.shape)
print('positivos:', df['houve_foco_d1'].sum())
print('taxa:', f"{100*df['houve_foco_d1'].mean():.3f}%")

## Distribuição temporal dos focos

O RN tem padrão sazonal forte. Esperamos pico no segundo semestre (set–dez), quando o sertão atinge o ápice da seca.

In [ ]:
diaria = df.groupby(df['data'].dt.date)['n_focos'].sum()
diaria.index = pd.to_datetime(diaria.index)
fig, ax = plt.subplots(figsize=(11, 4))
diaria.rolling(7).mean().plot(ax=ax, color='#b91c1c')
ax.set_title('focos diários (media movel 7 dias) - RN')
ax.set_xlabel('data'); ax.set_ylabel('n_focos')
plt.show()

mensal = df.groupby(df['data'].dt.month)['n_focos'].sum()
fig, ax = plt.subplots(figsize=(8, 3.5))
mensal.plot(kind='bar', ax=ax, color='#ea580c')
ax.set_title('focos por mes (somados 2020-2024)')
ax.set_xlabel('mes'); ax.set_ylabel('n_focos')
plt.show()

## Distribuição espacial

Quais municípios concentram focos? A região do sertão e do agreste do RN (centro-oeste do estado) costuma liderar.

In [ ]:
por_mun = df.groupby('nome_municipio')['n_focos'].sum().sort_values(ascending=False)
top = por_mun.head(15)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top.index[::-1], top.values[::-1], color='#b91c1c')
ax.set_title('top 15 municipios em focos (2020-2024)')
plt.tight_layout()
plt.show()

print('total municipios com pelo menos 1 foco:', (por_mun > 0).sum())
print('total municipios sem nenhum foco em 5 anos:', (por_mun == 0).sum())

## Correlação com variáveis meteorológicas

Esperamos correlação positiva com temperatura, FWI e DC (índice de seca de longo prazo); negativa com umidade e chuva acumulada.

In [ ]:
cols = ['temp_media', 'umid_media', 'chuva_dia', 'chuva_acum_7d', 'chuva_acum_30d',
        'dias_sem_chuva', 'vento_medio', 'fwi', 'ffmc', 'dc', 'houve_foco_d1']
corr = df[cols].corr(method='spearman')['houve_foco_d1'].drop('houve_foco_d1').sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(corr.index, corr.values, color=['#15803d' if v < 0 else '#b91c1c' for v in corr.values])
ax.axvline(0, color='k', linewidth=0.8)
ax.set_title('correlacao Spearman com houve_foco_d1')
plt.tight_layout()
plt.show()

## Desempenho dos modelos

Lê o arquivo de métricas salvo pela pipeline de treino.

In [ ]:
with open(os.path.join(RAIZ, 'backend/modelo/avaliacao/metricas.json')) as f:
    metricas = json.load(f)

linhas = []
for nome, r in metricas.items():
    if nome.startswith('_'):
        continue
    t = r['teste']
    linhas.append({'modelo': nome, 'AUC': t['auc'], 'AP': t['ap'], 'precisao': t['precision'],
                   'recall': t['recall'], 'f1': t['f1'], 'acc': t['acc']})
pd.DataFrame(linhas).round(3)

In [ ]:
# top features do random forest
imp = pd.Series(metricas['random_forest']['importancias']).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 6))
imp.tail(15).plot(kind='barh', ax=ax, color='#2563eb')
ax.set_title('top 15 features - random forest')
plt.tight_layout()
plt.show()

## Limites e discussão honesta

Pontos que afetam a leitura dos resultados:

**Cobertura desigual de estações.** São apenas 9 estações INMET no RN (8 efetivas após 2020 — uma saiu de operação) e o Arquipélago de São Pedro e São Paulo aparece como estação do RN apesar de estar a ~1000km da costa. Filtramos por bounding box continental e adicionamos PB e CE para reforçar interpolação nas divisas, mas grandes municípios do interior podem estar a 60-80km da estação mais próxima — o IDW vai suavizar muita variação local.

**Volume baixíssimo de positivos.** ~2841 focos em 5 anos diluem em 305 mil pares município-dia: 0.54% positivos. Métricas de acurácia são enganosas; AUC, recall e AP são as honestas. A AP fica baixa (~0.04) porque para cada positivo capturado o modelo gera muitos falsos positivos — é o preço de operar em alto recall sob extremo desbalanceamento.

**Vazamento temporal evitado.** Todo split é por data e features lag/acumuladas usam `shift(1)` antes da janela rolante — o valor do dia atual nunca contamina sua própria previsão.

**Persistência climática como proxy do futuro.** No modo "futuro" da UI, propagamos a meteorologia do último dia adiante. Para D+1 isso é razoável; para D+7 é uma simulação grosseira (sem entrada de previsão numérica de tempo). O aviso na UI deixa isso explícito.

**Viés sazonal forte.** O modelo aprendeu, e usa muito, doy_sin/doy_cos. Isso é informativo mas significa que mudanças climáticas ou anos atípicos podem degradar bastante a performance.

**Bioma simplificado.** Aproximamos o RN como caatinga predominante + faixa leste de mata atlântica, dividido por longitude -35.2°. O ideal seria sobrepor com shapefile oficial do IBGE/IBAMA de biomas, ficou pendente.


## Conclusões

- O **Random Forest balanceado** entrega AUC 0.78 e recall 0.34 no conjunto de teste (último semestre de 2024). É o melhor compromisso entre cobertura de positivos e taxa de falsos positivos para o caso de uso.
- O LightGBM ficou abaixo do RF em recall, apesar de AUC competitivo. Provavelmente se beneficiaria de mais ajuste de hiperparâmetros e janela de treino maior.
- Sem balanceamento, ambos modelos colapsam para sempre prever 0 (recall ~0). Confirma que `class_weight='balanced'` / `scale_pos_weight` são obrigatórios neste problema.
- As features mais importantes são sazonais (dia do ano em seno/cosseno), geográficas (longitude, latitude, distância do litoral, área) e meteorológicas acumuladas (chuva_acum_30d, DC). FWI cru aparece menor; o sinal dele está bem distribuído entre seus componentes (FFMC, DMC, DC).
- O sistema é **útil como ferramenta de priorização** (ranquear municípios de risco para alocação de equipes) e não como gatilho binário de evacuação. A taxa baixa de precisão impede uso isolado para decisão automática.